# Error Case 18 — 500: Concurrent Modification Conflict

**Error:** `500 Internal Server Error`  
**Root cause:** Two ETL jobs modify same catalog entity simultaneously → entity_version optimistic lock conflict  
**Fix:** Serialize catalog updates, implement retry logic

## Flow
```
Job A reads catalog: entity_version=5
Job B reads catalog: entity_version=5
Job A updates: WHERE entity_version=5 → success → entity_version=6
Job B updates: WHERE entity_version=5 → FAILS (now version=6)
  → EntityVersionMismatchException → 500
```

## Why this is different from Monday incident
```
Monday incident:    PostgreSQL restore → DB version < Polaris cache
This case:          Two concurrent writers → race condition
Monday fix:         Pod restart (clear cache)
This fix:           Retry with backoff (re-read version)
```

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD
import threading, time

cleanup()
print('✅ Setup complete')

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🧹 Cleaning up all test resources...
  ✅ [204] namespace: sqlclone0_catalog.ns1
  ✅ [204] namespace: sqlclone0_catalog.ns2
  ✅ [204] namespace: sqlclone10_catalog.ns1
  ✅ [204] namespace: sqlclone10_catalog.ns2
  ✅ [204] namespace: sqlclone11_catalog.ns1
  ✅ [204] namespace: sqlclone11_catalog.ns2
  ✅ [204] namespace: sqlclone12_catalog.ns1
  ✅ [204] namespace: sqlclone12_catalog.ns2
  ✅ [204] namespace: sqlclone13_catalog.ns1
  ✅ [204] namespace: sqlclone13_catalog.ns2
  ✅ [204] namespace: sqlclone14_catalog.ns1
  ✅ [204] namespace: sqlclon

KeyboardInterrupt: 

In [2]:
# Cell 2: Setup
r1 = create_catalog()
assert r1.status_code == 201, r1.text
print('Setup: catalog ✅')

# Get initial entity_version
r_get = requests.get(f'{BASE_MGMT}/catalogs/{TEST_CATALOG}', headers=h())
initial_version = r_get.json().get('entityVersion')
print(f'Initial entityVersion: {initial_version}')

Setup: catalog ✅
Initial entityVersion: 1


In [3]:
# Cell 3: Simulate concurrent modification
# Two jobs simultaneously try to update the same catalog
results = []
lock = threading.Lock()

def update_catalog_job(job_id, new_property):
    """Simulate ETL job updating catalog properties."""
    # Both jobs read catalog at same time
    time.sleep(0.1 * job_id)  # slight stagger to simulate real scenario
    
    r = requests.put(
        f'{BASE_MGMT}/catalogs/{TEST_CATALOG}',
        headers=h(),
        json={
            'catalog': {
                'name': TEST_CATALOG,
                'properties': {
                    'default-base-location': f's3a://data-catalog-bucket/{TEST_CATALOG}/',
                    'job-property': new_property,
                }
            }
        }
    )
    with lock:
        results.append({
            'job': job_id,
            'status': r.status_code,
            'message': r.json().get('error', {}).get('message', 'ok')[:80]
                       if r.status_code != 204 else 'ok'
        })

print('Launching 3 concurrent catalog update jobs...')
threads = [
    threading.Thread(target=update_catalog_job, args=(i, f'value-from-job-{i}'))
    for i in range(3)
]
for t in threads: t.start()
for t in threads: t.join()

print('\nResults:')
for r in results:
    icon = '✅' if r['status'] in [200, 204] else '❌'
    print(f'  {icon} Job {r["job"]}: {r["status"]} — {r["message"]}')

Launching 3 concurrent catalog update jobs...

Results:
  ❌ Job 0: 500 — Cannot invoke "java.lang.Integer.intValue()" because the return value of "org.ap
  ❌ Job 1: 500 — Cannot invoke "java.lang.Integer.intValue()" because the return value of "org.ap
  ❌ Job 2: 500 — Cannot invoke "java.lang.Integer.intValue()" because the return value of "org.ap


In [ ]:
# Cell 4: Find entity_version mismatch logs
wait_for_logs(10)
print('Searching for entity_version mismatch errors...')
hits = search_entity_version_errors(minutes_ago=2)
if not hits:
    hits = search_500_errors(minutes_ago=2)
print_logs(hits)

In [ ]:
# Cell 5: Fix — retry with backoff
import random

def update_catalog_with_retry(new_property, max_retries=3):
    """Update catalog with optimistic lock retry."""
    for attempt in range(max_retries):
        r = requests.put(
            f'{BASE_MGMT}/catalogs/{TEST_CATALOG}',
            headers=h(),
            json={
                'catalog': {
                    'name': TEST_CATALOG,
                    'properties': {
                        'default-base-location': f's3a://data-catalog-bucket/{TEST_CATALOG}/',
                        'updated-by': new_property,
                    }
                }
            }
        )
        if r.status_code in [200, 204]:
            print(f'  ✅ Attempt {attempt+1}: success')
            return r
        elif r.status_code == 500 and 'version' in r.text.lower():
            wait = (2 ** attempt) + random.uniform(0, 1)  # exponential backoff
            print(f'  ⚠️  Attempt {attempt+1}: version conflict — retry in {wait:.1f}s')
            time.sleep(wait)
        else:
            print(f'  ❌ Attempt {attempt+1}: {r.status_code} — not retryable')
            return r
    print(f'  ❌ All {max_retries} attempts failed')
    return None

# Test retry with concurrent jobs
print('Testing retry pattern with concurrent updates...')
retry_results = []

def job_with_retry(job_id):
    r = update_catalog_with_retry(f'job-{job_id}')
    with lock:
        retry_results.append({'job': job_id, 'success': r and r.status_code in [200, 204]})

threads = [threading.Thread(target=job_with_retry, args=(i,)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

print(f'\nAll jobs succeeded: {all(r["success"] for r in retry_results)}')

cleanup()
print('\n=== Summary ===')
print('Error:    500 EntityVersionMismatch')
print('Cause:    Concurrent ETL jobs update same entity')
print('Detect:   search_entity_version_errors()')
print('Fix:      Retry with exponential backoff (2^attempt + jitter)')
print('Pattern:  Read entity → modify → write → if 500: re-read → retry')
print('Prevent:  Serialize catalog updates with distributed lock')